# Optimization under uncertainty: DSIVC and the decision space

_What if we pumped differently?_

Everything up to here has answered a single question about a single plan: given how the supply well (`wellopt`) is scheduled to run, *how much* sulfate will the supplied water carry, and how sure are we? The history match sharpened that forecast distribution; the dataworth notebook asked which extra measurements would sharpen it further. None of it told us what to *do*.

This notebook closes the loop. The supply well is not a fixed object. It draws from **three screens stacked vertically** in the aquifer (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`), and a manager can set *how hard each screen pumps independently* — the **vertical distribution** of extraction — as well as *when* in the supply period the well switches on. Those are the levers actually available at the wellhead. We pose them as **decision variables** and search for the best trade-off between supplying more water and keeping peak sulfate low — a bi-objective **optimization under uncertainty** — using **DSIVC** ("DSI variable control"), the emulator-driven optimizer that ships with the `feat_dsivc` branch of `pyemu`.

Why the *vertical* distribution is worth posing: the injected oxic water and the sulfate it liberates from the pyrite are not uniform with depth. Some screens sit closer to the redox front than others, so they draw hotter water. Pumping every screen in lockstep hides that structure; letting the screens move independently *could* give the optimizer a genuine lever — back off a hot screen, lean on the clean ones. Whether it actually *does* — whether that lever creates a real volume-versus-quality trade-off or the objectives simply align — is exactly what the search is for. We will pose it honestly and read whatever the emulator and the physics return.

Where this sits in the sequence:

- [`../part1_05_dsi_basics/dizon_dsi_basics.ipynb`](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) trained, validated and conditioned the DSI emulator — the machinery and the discipline ("never trust an emulator you have not tested") carry straight into this notebook.
- [`../part1_06_full_model_check/`](../part1_06_full_model_check/) validated the DSI posterior against a genuine full-model history match.
- [`../part1_07_dataworth/`](../part1_07_dataworth/) used the emulator to value held-back *data*.
- This notebook is the last in the sequence: it uses the same emulation-first thinking to value held-back *decisions*.

### Admin

We lean on the vendored dependency trees (`flopy`, `pyemu`) that ship with this repository — in particular the `rhugman/pyemu@feat_dsivc` branch, which carries the `DSIVC` workflow that is the whole subject of this notebook. As elsewhere in the series, `herebedragons` (imported as `hbd`) holds the shared plotting and bookkeeping helpers, `hbd.get_bins` copies the platform binaries (`pestpp-ies`, `pestpp-mou`, …) into each workspace, and every workspace this notebook creates lives *inside this notebook's own directory* and is gitignored.

One prebaked artifact does the heavy lifting: the **decision-variable training sweep** (`../../prebaked/dsivc_sweep/`). What it is, why it had to be run, and what it cost are spelled out in [the sweep section](#The-training-sweep:-why-plain-DSI-is-not-enough) below. Everything downstream of the sweep — fitting the emulator, wiring up DSIVC, and running the multi-objective search — happens live in this notebook, on the emulator alone, with **no full-model runs in the loop**.

In [ ]:
import os
import sys
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import flopy
import pyemu

warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
import herebedragons as hbd

Confirm we are running the vendored dependencies, and that this `pyemu` carries the DSIVC workflow (it lives only on the `feat_dsivc` branch we vendor):

In [ ]:
assert "dependencies" in flopy.__file__, flopy.__file__
assert "dependencies" in pyemu.__file__, pyemu.__file__
from pyemu.emulators import DSI, DSIVC   # DSIVC exists only on rhugman/pyemu@feat_dsivc
assert hasattr(pyemu.emulators, "DSIVC"), (
    "this pyemu has no DSIVC; refresh dependencies/pyemu from rhugman/pyemu@feat_dsivc"
)
print("vendored pyemu with DSIVC present")

**Prerequisite check.** This notebook consumes one prebaked product: the decision-variable training sweep. It ships as two files in `../../prebaked/dsivc_sweep/`, read against the thinned curated-obs control file `../../prebaked/pest.pst`:

- `sweep_obs_ensemble.jcb` — the model outputs (curated conditioning + forecast observations) for every surviving sweep run;
- `sweep_dvs.csv` — the decision-variable values (per-screen rate multipliers and switch-on day) that produced each run.

If the sweep is missing, the cell below stops with a pointer to the maintainer script that bakes it.

In [ ]:
prebaked = Path("..") / ".." / "prebaked"
sweep_dir = prebaked / "dsivc_sweep"
sweep_oe_path = sweep_dir / "sweep_obs_ensemble.jcb"
sweep_dv_path = sweep_dir / "sweep_dvs.csv"
pst_path = prebaked / "pest.pst"

for p in (sweep_oe_path, sweep_dv_path, pst_path):
    if not p.exists():
        raise Exception(
            f"required prebaked artifact not found: {p}\n"
            "the DSIVC training sweep is baked by etc/run_dsivc_sweep.py followed by "
            "etc/make_prebaked.py dsivc-sweep --source master_sweep")
print("prebaked sweep artifacts found")

## The decision problem

Four **decision variables** describe how the supply well is operated. Three of them set the **vertical distribution** of pumping — one multiplier per screen — and the fourth sets the timing:

- **`dv-rate-ly1`, `dv-rate-ly3`, `dv-rate-ly5`** — a multiplier on each screen's base extraction rate (base rates −1300 / −300 / −300 m³/d), each drawn across `[0, 2]`. A value of `0` closes that screen entirely; `2` doubles it. Because the screens move independently, the optimizer can *reshape* the well's vertical draw, not just scale it.
- a **switch-on day** within the supply period — *when* we start, drawn across `[308, 500]` d. The well activates at the first stress period whose start is on or after the switch day (the model schedules pumping at stress-period granularity; the continuous switch-day is a relaxation the search smooths over).

Two **objectives** describe what we care about:

- **maximize the volume of water supplied** over the supply period, and
- **minimize P95(delivered blend SO₄)** — the 95th percentile, across the residual parameter-field uncertainty, of the peak sulfate concentration in the *flow-weighted blend* the supply well actually delivers (defined in "The forecast" below). This — not the worst single screen — is what the treatment plant treats, and it is the quantity the vertical-distribution decision can move.

There is no *a priori* single best plan; DSIVC will search the decision space and hand us the set of non-dominated trade-offs — the "front" — and we read it honestly, whatever shape it turns out to have.

The objectives are **minimize-native**: lower P95(delivered blend SO₄), more volume. No threshold is baked into the search, and none is needed. Choosing P95 (rather than the mean) is the decision-support move: the manager is not asked "what is the expected blend?" but "how much water can I promise, and how high might the delivered sulfate run while I do?" The percentile *is* the risk appetite, stated out loud — treatment capacity is sized to it.

A contractual trigger, if one exists, is an **illustrative lens** drawn across the same front, not a second analysis: with a 90 mg/L supply-contract trigger on the sulfate axis, the *reliable* (chance-constrained) reading is simply the segment of the front below the line. (The EU drinking-water standard of 250 mg/L is comfortably met across the whole front; cost, not compliance, drives this decision.)

A unit note worth stating once. The forecast obs (`welopt-ly*`, variable `so4`) are carried in **mol/L**, the way the transport model writes them. A contractual trigger such as 90 mg/L is in mg/L. The conversion uses the molar mass of SO₄ (≈96.06 g/mol), so 90 mg/L ≈ `90e-3 / 96.06` mol/L. We carry the ensemble in mg/L for display and convert the trigger the same way, so the figures speak one language.

In [ ]:
# supply-period bookkeeping (constants only; no model run).
SO4_MW = 96.06                      # g/mol
CONTRACT_TRIGGER_MGL = 90.0         # illustrative supply-contract trigger (lens only)

SUPPLY_START, SUPPLY_END = 308.0, 728.0   # supply period (days); the forecast lives here
DECISION_DATE = 252.0                      # decision committed ~8 weeks before switch-on

# the three supply screens and their base extraction rates (m3/d), and the
# decision variable that scales each. dv name -> screen key -> base rate.
SCREEN_BASE = {"ly1": 1300.0, "ly3": 300.0, "ly5": 300.0}
RATE_DVS = [f"dv-rate-{k}" for k in SCREEN_BASE]        # per-screen multipliers
DECVARS = RATE_DVS + ["dv-switch-day"]                  # + switch-on timing
BASE_RATE_TOTAL = sum(SCREEN_BASE.values())            # 1900 m3/d at all-screens-1.0

# decision-variable bounds (the sweep was drawn across exactly these)
RM_LO, RM_HI = 0.0, 2.0             # per-screen rate multiplier
SW_LO, SW_HI = 308.0, 500.0         # switch-on day

# the supply stress periods and their start days, read from the sweep design
# (run_dsivc_sweep.py): SP index -> start day. The last SP ends at SUPPLY_END.
SUPPLY_SP_STARTS = {22: 308, 23: 336, 24: 364, 25: 392, 26: 420, 27: 455, 28: 490,
                    29: 518, 30: 546, 31: 574, 32: 609, 33: 644, 34: 672, 35: 700}

# the canonical operation, used throughout as the reference plan: every screen at
# its base rate (multiplier 1.0), switching on at the start of the supply window.
CANONICAL = {**{d: 1.0 for d in RATE_DVS}, "dv-switch-day": SUPPLY_START}

print(f"90 mg/L  ->  {CONTRACT_TRIGGER_MGL * 1e-3 / SO4_MW:.4e} mol/L")
print(f"supply window {SUPPLY_START:.0f}-{SUPPLY_END:.0f} d")
print(f"decision variables ({len(DECVARS)}): {DECVARS}")
print(f"per-screen base rates (m3/d): {SCREEN_BASE}  ->  total {BASE_RATE_TOTAL:.0f}")

## The training sweep: why plain DSI is not enough

Here is the catch, and it is the reason this notebook needs its own prebaked artifact.

DSI is an **observation-space emulator**. In `part1_05` it learned the joint distribution of the model outputs — the breakthrough series at every site and species, including the supply-well sulfate forecast — *as produced by the prior parameter ensemble running one fixed supply-well schedule*. Conditioning it with PESTPP-IES slid that distribution toward the data. At no point did that emulator see a run where the pump rate or switch-on day was different, because no such run exists in the prior Monte Carlo. The decision variables are **inputs to the forward model**, not outputs the emulator ever observed.

So the `part1_05` emulator cannot answer "what if we pumped differently?" by interpolation. It has no axis for the decision. Asking it to extrapolate a pumping change it never saw would be exactly the kind of unvalidated emulator use the [fidelity-check beat](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) warned against.

The fix is to give the emulator that axis on purpose. **Coverage of decision space is designed, not inherited.** A dedicated **training sweep** takes the history-matched *posterior* parameter fields (from the full-model IES of `part1_06`) and, for each one, re-samples the decision variables across their bounds and runs the full model. Each run is a (posterior field, pumping plan) pair. The sweep thus spans both the residual parameter uncertainty *and* the decision space — and the decision variables now appear as values attached to each run, alongside the outputs. That is the raw material DSIVC needs.

Let us load the sweep and read the provenance off the sidecar:

In [ ]:
import json
with open(sweep_dir / "dsivc_sweep.json") as f:
    prov = json.load(f)
for k in ("source_dir", "n_realisations", "thinned_to_n_obs", "cost_note", "notes"):
    print(f"{k}: {prov[k]}")

In [ ]:
# the control file for the curated obs set, and the sweep obs ensemble against it.
pst = pyemu.Pst(str(pst_path))
pst.try_parse_name_metadata()
oe = pyemu.ObservationEnsemble.from_binary(pst=pst, filename=str(sweep_oe_path))
sweep_obs = oe._df.copy()
sweep_obs.index = sweep_obs.index.astype(str)

# the decision-variable values that produced each run (join on the run index)
dvs = pd.read_csv(sweep_dv_path, index_col=0)
dvs.index = dvs.index.astype(str)

# the runs we can actually use: obs and dv values both present
run_ids = [i for i in sweep_obs.index if i in dvs.index]
print(f"sweep obs ensemble: {sweep_obs.shape[0]} runs x {sweep_obs.shape[1]} obs")
print(f"usable (posterior field x decision draw) runs: {len(run_ids)}")
print(f"decision variables: {list(dvs.columns)}")

## The forecast: what the supply well actually *delivers*

Everywhere else in this series the reported forecast is **peak SO₄ = the maximum over the three supply screens and over the supply-window times** — a deliberately conservative *monitoring* metric: it flags the worst water any single screen sees. That is the right lens for "could this well ever produce out-of-spec water?".

But for an **operational decision** it is the wrong quantity, and this is the crux of the whole notebook. The three screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) do not deliver three separate waters — they feed **one pipe**. What the treatment plant treats, and what the customer drinks, is the **flow-weighted blend**:

$$C_\text{blend}(t) \;=\; \frac{\sum_s q_s\, C_s(t)}{\sum_s q_s}, \qquad q_s = (\text{base rate})_s \times (\text{multiplier})_s$$

and the forecast we optimize is the **peak of that blend** over the times the well is actually delivering (stress periods on or after the switch-on day). This is the quantity the *vertical distribution* decision can actually move: throttle a screen that runs hot in adverse aquifer states and its high-sulfate water is a smaller share of the blend — at the cost of the volume that screen would have contributed. A maximum-over-screens forecast cannot be moved that way (reallocating between screens never lowers a maximum); a blend can. We keep the worst-screen peak in view as the conservative monitor, but the **objective is the delivered blend**.

In [ ]:
obs = pst.observation_data
fore = obs.loc[obs.obgnme == "forecast"].copy()
fore_cols = [c for c in fore.obsnme if c in sweep_obs.columns]
fore["time"] = pd.to_numeric(fore["time"])
print(f"forecast group: {len(fore_cols)} obs "
      f"(screens {sorted(fore.obsid.unique())}, variable {sorted(fore.variable.unique())}, "
      f"times {fore.time.min():.0f}-{fore.time.max():.0f} d)")

# per-screen breakthrough columns, ordered on a shared time grid, for blending
SCREENS = ["welopt-ly1", "welopt-ly3", "welopt-ly5"]
_byt = {s: fore.loc[fore.obsid == s].sort_values("time") for s in SCREENS}
FORE_TIME = _byt[SCREENS[0]]["time"].values                       # shared time grid
SCR_COLS = {s: list(_byt[s]["obsnme"]) for s in SCREENS}           # cols in time order

def plan_rates(plan):
    """Per-screen extraction rate q_s (m3/d) for a plan dict."""
    return {s: SCREEN_BASE[s.split("-")[-1]] * plan[f"dv-rate-{s.split('-')[-1]}"]
            for s in SCREENS}

def worst_peak_mgl(ensemble):
    """Conservative monitor: peak SO4 (mg/L), max over ALL screens, per row."""
    cols = [c for c in fore_cols if c in ensemble.columns]
    return ensemble.loc[:, cols].max(axis=1) * 1000.0 * SO4_MW

def blend_peak_mgl(ensemble, plan):
    """Delivered forecast: peak of the flow-weighted blend (mg/L) over the times
    the well is on, per row, at this plan's per-screen rates."""
    q = plan_rates(plan)
    tot = sum(q.values())
    if tot <= 1e-9:
        return pd.Series(np.nan, index=ensemble.index)
    on = FORE_TIME >= plan["dv-switch-day"]
    blend = sum(q[s] * ensemble.loc[:, SCR_COLS[s]].values for s in SCREENS) / tot
    return pd.Series((blend[:, on] * 1000.0 * SO4_MW).max(axis=1), index=ensemble.index)

# each sweep run, blended at ITS OWN decisions -> a valid (decision -> delivered) sample
_run_plan = {r: {**{f"dv-rate-{s.split('-')[-1]}":
                    float(dvs.loc[r, f"dv-rate-{s.split('-')[-1]}"]) for s in SCREENS},
                 "dv-switch-day": float(dvs.loc[r, "dv-switch-day"])} for r in run_ids}
blend = pd.Series({r: blend_peak_mgl(sweep_obs.loc[[r]], _run_plan[r]).iloc[0]
                   for r in run_ids})
worst = worst_peak_mgl(sweep_obs).loc[run_ids]
print(f"\ndelivered BLEND peak across sweep (mg/L): min={blend.min():.0f}  "
      f"median={blend.median():.0f}  P95={blend.quantile(0.95):.0f}  max={blend.max():.0f}")
print(f"worst-SCREEN peak (monitor)   (mg/L): min={worst.min():.0f}  "
      f"median={worst.median():.0f}  P95={worst.quantile(0.95):.0f}  max={worst.max():.0f}")

## How DSIVC works

The vendored `pyemu` carries a purpose-built workflow for exactly this problem: **DSIVC**, "DSI variable control". Its central idea is elegant. Treat the decision variables as **observations** in DSI-world — columns of the training ensemble that a manager can *control* rather than merely observe. Fit a DSI emulator on `[model outputs + decision-variable columns]`, then wrap an outer PESTPP-MOU optimizer around it. Evaluating one candidate plan becomes a nested loop:

1. the outer optimizer (PESTPP-MOU) proposes a plan — a `(rate_ly1, rate_ly3, rate_ly5, switch-on day)` vector;
2. those values are injected into the inner `dsi.pst` as **high-weight, zero-noise observation targets** — "condition the emulator on the world where we pump *this* way";
3. a nested PESTPP-IES conditioning runs against the emulator in run-storage mode (`pestpp-ies dsi.pst /e`), producing a conditioned posterior ensemble — the **"stack"**;
4. the stack is summarized into per-output **percentiles** (the "stack stats") — and *those* percentiles become the outer problem's objectives and constraints.

So P95(delivered blend peak) for a candidate plan is not read off a fixed table — it is *recomputed* by conditioning the emulator on that plan, every time the optimizer asks. The rest of this notebook builds that machine, evaluates it by hand once to see it turn, probes the surface it defines, and then lets PESTPP-MOU search it.

### Step 1 — the DSIVC training ensemble

DSIVC wants one ensemble whose columns are `[the outputs we care about] + [the decision variables]`. We do not need the history-window conditioning obs here: the sweep already ran on *posterior* (history-matched) fields, so the inner conditioning slides the ensemble along the *decision* axes only, not the data axes. We assemble a compact training frame from the per-screen forecast breakthrough series, two derived forecast columns — the **delivered blend peak** (the thing we minimize the P95 of) and the **worst-screen peak** (the conservative monitor) — and the four decision variables from `sweep_dvs.csv`.

The blend-peak is an *uncertain* forecast (it depends on the aquifer parameters), so we hand it to the emulator as a derived column — exactly as the worst-screen peak is handled — and let DSI carry it jointly with the per-screen series and the decision variables. When the optimizer later conditions on a candidate's rates, the emulated blend-peak shifts with the allocation. (Volume, by contrast, is *deterministic*; it is computed exactly later, never emulated.)

In [ ]:
# per-screen forecast breakthrough series (the joint structure blend/peak draw from)
train = sweep_obs.loc[run_ids, fore_cols].copy()

# derived forecast columns (stored in mol/L, the model's native unit, like the rest):
#  - delivered blend peak, each run blended at ITS OWN decisions (the objective)
#  - worst-screen peak, max over screens (the conservative monitor, kept for context)
train["fore-blend-peak"] = (blend.loc[run_ids] / (1000.0 * SO4_MW)).values
train["fore-peak-so4"] = sweep_obs.loc[run_ids, fore_cols].max(axis=1)

# the decision variables, attached as columns -> DSIVC will treat these as the
# "observations a manager controls" (three per-screen rates + the switch-on day)
for d in DECVARS:
    train[d] = dvs.loc[run_ids, d].astype(float)

print(f"training ensemble: {train.shape[0]} runs x {train.shape[1]} columns "
      f"({len(fore_cols)} forecast + blend-peak + worst-peak + {len(DECVARS)} decisions)")

### Step 2 — fit the DSI emulator (now with decision axes)

This is the same DSI machinery from `part1_05` — a normal-score transform, an SVD to a compact latent space — but fitted on the sweep, so the latent space now carries the decision variables alongside the outputs. The emulator learns the *joint* distribution of `(outputs, peak, rate, switch)` across the sweep.

In [ ]:
transforms = [{"type": "normal_score", "quadratic_extrapolation": True}]
dsi = DSI(data=train, transforms=transforms, energy_threshold=0.99)
dsi.fit()
print(f"DSI fitted: {dsi.latent_dim} latent dimensions "
      f"(from {train.shape[0]} training runs)")

### Step 3 — prepare the run-storage DSI interface

DSIVC drives the inner conditioning in **run-storage mode** (`pestpp-ies dsi.pst /e`): PESTPP-IES writes every realisation's latent coordinates into a single run-storage file, the emulator predicts them all in one batch, and the outputs are written straight back — no per-realisation process launches. We prepare that interface with `prepare_pestpp(..., use_runstor=True)` and drop the platform binaries into the template so the nested runs find a local `pestpp-ies`:

In [ ]:
dsi_td = Path("dsi_template")
dpst = dsi.prepare_pestpp(str(dsi_td), use_runstor=True)
hbd.get_bins(str(dsi_td))

# sanity: the runstore forward run must be wired, and the decision variables must
# be present as (currently zero-weight) observations DSIVC can later target
frun = (dsi_td / "forward_run.py").read_text()
assert "dsi_runstore_forward_run" in frun, "DSI template is not runstore-prepared"
assert all(d in dpst.obs_names for d in DECVARS), "decision variables missing from dsi.pst obs"
print(f"runstore DSI interface ready: {dpst.nobs} obs, {dpst.npar} latent parameters")

### Step 4 — construct the DSIVC workflow

`DSIVC` takes the fitted emulator, the run-storage template directory, and an observation ensemble whose columns **equal** the emulator's observation set exactly (decision variables included). That ensemble is just our training frame wrapped as a `pyemu.ObservationEnsemble` against the inner `dsi.pst`:

In [ ]:
dsi_pst = pyemu.Pst(str(dsi_td / "dsi.pst"))
oe_dsivc = pyemu.ObservationEnsemble(pst=dsi_pst, df=train.copy())

dsivc = DSIVC(emulator=dsi, dsi_t_d=str(dsi_td), oe=oe_dsivc, verbose=False)
print("DSIVC constructed (emulator + runstore template + matching obs ensemble)")

### Step 5 — build the outer PESTPP-MOU interface

`prepare_pestpp` copies the DSI template into a fresh directory, wires the decision variables in as the inner conditioning targets, and stages the **stack-stats** observations — one per `(output, percentile)` pair — that the outer optimizer will read. We ask for the 5th/50th/95th percentiles of every output, so the P95 of the delivered blend peak is available as an objective. `inner_noptmax` is how many iterations the nested PESTPP-IES conditioning runs per candidate.

In [ ]:
mou_td = Path("dsivc_template")
mpst = dsivc.prepare_pestpp(
    str(mou_td),
    decvar_names=DECVARS,
    percentiles=[0.05, 0.5, 0.95],
    inner_noptmax=3,
    mou_population_size=20,
)
hbd.get_bins(str(mou_td))

# the risk objective lives among the stack-stats: the 95th percentile of the
# delivered BLEND-peak column (the worst-screen peak is also staged, as a monitor)
ss = mpst.observation_data.loc[mpst.observation_data.obgnme == "stack_stats"]
blend_p95_obs = "fore-blend-peak_stat:95%"
assert blend_p95_obs in ss.index, sorted(c for c in ss.index if "blend" in c)
print(f"outer MOU interface: {mpst.nobs} obs "
      f"({len(ss)} stack-stats over {mpst.npar_adj} decision variables)")
print(f"risk objective staged as: {blend_p95_obs}")

### Step 6 — the two objectives (and why volume must *not* be emulated)

The risk objective, **P95(delivered blend peak)**, is genuinely uncertain — it depends on the aquifer parameters we never fully pinned down — so it belongs to the emulator: `fore-blend-peak_stat:95%`, read off the conditioned stack.

The volume objective is different. Volume supplied is a **deterministic, known** function of the decisions — switch-on day sets which stress periods are active, and each screen's multiplier scales that screen's contribution. There is no uncertainty to emulate, and pushing a deterministic quantity through the DSI/Gaussian conditioning actually *corrupts* it (the linear-Gaussian update smears a value we know exactly). So we compute volume **outside** the emulator, as a second model command that PEST++ runs alongside the DSIVC conditioning, and register its output as an ordinary observation. With per-screen rates the volume is a sum over screens: total instantaneous rate is `1300·m_ly1 + 300·m_ly3 + 300·m_ly5`, integrated over the active stress periods.

In [ ]:
# closed-form volume, written as a tiny standalone script PEST++ runs per candidate
volume_script = """import pandas as pd
_st = sorted([308, 336, 364, 392, 420, 455, 490, 518, 546, 574, 609, 644, 672, 700])
SP_LEN = {s: (_st[i + 1] if i + 1 < len(_st) else 728.0) - s for i, s in enumerate(_st)}
BASE = {"ly1": 1300.0, "ly3": 300.0, "ly5": 300.0}
d = pd.read_csv("dsivc_pars.csv", index_col=0).iloc[:, 0]
sw = float(d["dv-switch-day"])
rate = sum(BASE[k] * float(d["dv-rate-" + k]) for k in BASE)   # total m3/d this plan
vol = sum(L * rate for s, L in SP_LEN.items() if s >= sw)
with open("supply_vol.dat", "w") as f:
    f.write("supply_vol %.6e\\n" % vol)
"""
(mou_td / "compute_volume.py").write_text(volume_script)

# closed form we also use directly in this notebook (identical arithmetic)
_starts = sorted(SUPPLY_SP_STARTS.values())
SP_LENGTHS = {s: (_starts[i + 1] if i + 1 < len(_starts) else SUPPLY_END) - s
              for i, s in enumerate(_starts)}
def volume_supplied(plan):
    """m3 supplied by a plan dict {dv-rate-ly*: mult, dv-switch-day: day} =
    sum over supply SPs with start >= switch of (SP length * total plan rate)."""
    rate = sum(SCREEN_BASE[d.split("-")[-1]] * plan[d] for d in RATE_DVS)
    return sum(L * rate for s, L in SP_LENGTHS.items() if s >= plan["dv-switch-day"])

base_volume = volume_supplied(CANONICAL)
_all_max = {**{d: RM_HI for d in RATE_DVS}, "dv-switch-day": SUPPLY_START}
print(f"canonical operation (all screens 1.0, switch {SUPPLY_START:.0f}): {base_volume:,.0f} m3")
print(f"maximum (all screens {RM_HI}, switch {SUPPLY_START:.0f}): "
      f"{volume_supplied(_all_max):,.0f} m3")

Wire the exact-volume script in as a second model command, register its output as an observation, and declare the two objectives to PESTPP-MOU. Direction is set by the observation group name the optimizer reads: a `less_than` group is minimized, a `greater_than` group is maximized.

In [ ]:
# run the volume script once so the output file exists, then build its instruction file
pd.DataFrame({"parval1": [CANONICAL[d] for d in DECVARS]}, index=DECVARS).to_csv(
    mou_td / "dsivc_pars.csv")
pyemu.os_utils.run("python compute_volume.py", cwd=str(mou_td))
(mou_td / "supply_vol.dat.ins").write_text("pif ~\nl1 ~ ~ !supply-vol-exact!\n")

# register the exact-volume observation and add the 2nd model command
mpst.add_observations(str(mou_td / "supply_vol.dat.ins"),
                      str(mou_td / "supply_vol.dat"), pst_path=".")
mpst.model_command = ["python dsivc_forward_run.py", "python compute_volume.py"]

# declare objectives: minimize P95(delivered blend peak), maximize exact volume
obsd = mpst.observation_data
obsd.loc[blend_p95_obs, ["weight", "obgnme"]] = [1.0, "less_than"]
obsd.loc["supply-vol-exact", ["weight", "obgnme"]] = [1.0, "greater_than"]
mpst.pestpp_options["mou_objectives"] = f"{blend_p95_obs},supply-vol-exact"
mpst.pestpp_options["mou_generator"] = "de"
mpst.write(str(mou_td / "dsivc.pst"), version=2)
print("objectives declared:")
print(f"  minimize  {blend_p95_obs}   (delivered blend, from the conditioned DSI stack)")
print(f"  maximize  supply-vol-exact              (computed exactly, not emulated)")

## One candidate, evaluated by hand

Before we let the optimizer loose, let us turn the crank once and watch DSIVC work. We inject a single plan — the canonical operation, every screen at 1.0 switching on at day 308 — run the DSIVC forward model, and read back the **conditioned stack**: the full posterior forecast for *that plan*, not a point estimate. This is the object PESTPP-MOU summarizes into one objective value, thousands of times over.

In [ ]:
import subprocess

def evaluate_plan(plan, td=mou_td):
    """Inject one plan (a dict {decvar: value}), run the DSIVC forward model, and
    return the conditioned delivered-blend-peak stack-stats (mg/L) and the exact
    volume (m3). The nested PESTPP-IES chatter is captured (not printed)."""
    pd.DataFrame({"parval1": [plan[d] for d in DECVARS]}, index=DECVARS).to_csv(
        td / "dsivc_pars.csv")
    subprocess.run(["python", "dsivc_forward_run.py"], cwd=str(td),
                   capture_output=True, text=True, check=True)
    stats = pd.read_csv(td / "dsi.stack_stats.csv", index_col=0).iloc[:, 0]
    blend_stats = {q: float(stats[f"fore-blend-peak_stat:{q}"]) * 1000.0 * SO4_MW
                   for q in ("5%", "50%", "95%")}
    return blend_stats, volume_supplied(plan)

pk, vol = evaluate_plan(CANONICAL)
print(f"canonical plan (all screens 1.0, switch {SUPPLY_START:.0f}):")
print(f"  conditioned delivered blend SO4 (mg/L):  P5={pk['5%']:.1f}  "
      f"median={pk['50%']:.1f}  P95={pk['95%']:.1f}")
print(f"  volume supplied (exact):                 {vol:,.0f} m3")

That single P95 printed above is what one row of the optimizer's population costs: a full nested conditioning of the emulator, reduced to one percentile of the delivered blend. Because it runs against the emulator in run-storage mode, it takes a second or two — which is the whole reason a multi-objective search over hundreds of such evaluations is tractable at all.

## Probing the decision space: which screen moves the blend?

With four decision variables we cannot draw the response surface as a single contour. But before turning the optimizer loose we can ask a sharper question that motivates the whole vertical-distribution idea: **holding everything else at the canonical operation, what does moving one screen's rate do to the delivered blend?** We sweep each screen's multiplier across its bounds one at a time, condition DSIVC at every step, and read the P95 of the delivered blend peak. Three curves, one per screen — each the emulator's honest response to leaning on that screen alone. Because the objective is now a *blend*, reallocating between screens genuinely moves it (a maximum-over-screens forecast would sit flat here, as we saw was the trap).

In [ ]:
probe_grid = np.linspace(RM_LO, RM_HI, 6)
rows = []
for dv in RATE_DVS:
    for m in probe_grid:
        plan = {**CANONICAL, dv: m}
        pk, vol = evaluate_plan(plan)
        rows.append((dv, m, pk["95%"], pk["50%"], vol))
probe = pd.DataFrame(rows, columns=["screen_dv", "mult", "p95", "p50", "volume"])
print("conditioned P95(delivered blend) response, one screen swept at a time:")
for dv in RATE_DVS:
    sub = probe[probe.screen_dv == dv]
    print(f"  {dv:>13}: P95 spans {sub.p95.min():.1f}-{sub.p95.max():.1f} mg/L "
          f"as its multiplier goes {RM_LO:.1f}->{RM_HI:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5))
colors = {"dv-rate-ly1": "C3", "dv-rate-ly3": "C0", "dv-rate-ly5": "C2"}
for dv in RATE_DVS:
    sub = probe[probe.screen_dv == dv].sort_values("mult")
    ax.plot(sub.mult, sub.p95, marker="o", color=colors[dv],
            label=f"{dv}  (base {SCREEN_BASE[dv.split('-')[-1]]:.0f} m$^3$/d)")
ax.axhline(CONTRACT_TRIGGER_MGL, color="crimson", ls="--", lw=1.2,
           label=f"{CONTRACT_TRIGGER_MGL:.0f} mg/L trigger (illustrative)")
ax.axvline(1.0, color="0.6", ls=":", lw=1, label="canonical (all screens = 1.0)")
ax.set_xlabel("that screen's rate multiplier (others held at 1.0)")
ax.set_ylabel("DSIVC-conditioned P95(delivered blend SO$_4$) (mg/L)")
ax.set_title("Which screen moves the delivered blend? One-at-a-time response",
             loc="left", fontsize="small")
ax.legend(fontsize="small", loc="best")
fig.tight_layout()

Read the three curves. Pumping a screen harder does two competing things to the delivered blend: it adds that screen's water to the mix (raising the blend if that screen runs hot, lowering it if clean) *and* it raises the total flow, diluting the whole blend. The net slope, screen by screen, is what the emulator reports here — and it is the redistribution structure the optimizer will exploit. Where a curve rises, that screen is a net liability to delivered quality and a candidate to throttle; where it falls, leaning on that screen is a genuine free lunch. Which is which is computed, not assumed.

## The search: PESTPP-MOU over the emulator

Now we let the optimizer do the work the grid did by hand, but adaptively and in parallel. PESTPP-MOU maintains a population of candidate plans, evaluates each by conditioning the emulator (`pestpp-ies dsi.pst /e` inside every forward run), sorts them by Pareto dominance on our two objectives, and breeds the next generation toward the non-dominated front. We run a modest search — 20 plans, a handful of generations — across local workers. Every evaluation is emulator-only; **no full model runs here.**

In [ ]:
n_workers = max(2, psutil.cpu_count(logical=False) - 2)
mpst.control_data.noptmax = 8          # generations
mpst.write(str(mou_td / "dsivc.pst"), version=2)

master = Path("master_mou")
if master.exists():
    shutil.rmtree(master)
print(f"launching PESTPP-MOU: pop 20, {mpst.control_data.noptmax} generations, "
      f"{n_workers} workers (emulator-only) ...")
pyemu.os_utils.start_workers(
    str(mou_td), "pestpp-mou", "dsivc.pst",
    num_workers=n_workers, worker_root=".", master_dir=str(master),
)
print("PESTPP-MOU complete")

Read the final population and its non-dominated archive back in, and convert the delivered-blend objective to mg/L for display:

In [ ]:
dv_pop = pd.read_csv(master / "dsivc.dv_pop.csv", index_col=0)
obs_pop = pd.read_csv(master / "dsivc.obs_pop.csv", index_col=0)
arc_dv = pd.read_csv(master / "dsivc.archive.dv_pop.csv", index_col=0)
arc_obs = pd.read_csv(master / "dsivc.archive.obs_pop.csv", index_col=0)

def frame(dv, ob):
    out = pd.DataFrame({d: dv[d].values for d in DECVARS}, index=dv.index)
    out["p95_blend"] = ob[blend_p95_obs].values * 1000.0 * SO4_MW
    out["volume"] = ob["supply-vol-exact"].values
    return out

pop = frame(dv_pop, obs_pop)
archive = frame(arc_dv, arc_obs).sort_values("volume")
print(f"final population: {len(pop)} plans")
print(f"non-dominated archive: {len(archive)} plan(s)")
print(archive[DECVARS + ["p95_blend", "volume"]].to_string(
    float_format=lambda x: f"{x:,.1f}"))

## The result, read honestly

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.5))
# every plan the optimizer evaluated (the population), faint
ax.scatter(pop.volume / 1e3, pop.p95_blend, c="0.8", s=14, zorder=1,
           label="plans evaluated (MOU population)")
# the non-dominated archive
ax.scatter(archive.volume / 1e3, archive.p95_blend, c="crimson", s=90,
           edgecolor="k", lw=0.5, zorder=4, marker="D", label="non-dominated (Pareto)")
# the 90 mg/L contractual lens
ax.axhline(CONTRACT_TRIGGER_MGL, color="crimson", ls="--", lw=1.2, zorder=2,
           label=f"{CONTRACT_TRIGGER_MGL:.0f} mg/L contract trigger (illustrative)")
# canonical operation, for reference
base_pk, _ = evaluate_plan(CANONICAL)
ax.scatter([base_volume / 1e3], [base_pk["95%"]], marker="*", s=320, color="fuchsia",
           edgecolor="k", zorder=5, label="canonical operation")
ax.set_xlabel("volume supplied (thousand m$^3$)")
ax.set_ylabel("P95(delivered blend SO$_4$) (mg/L)")
ax.set_title("Supply-well operation under uncertainty: what DSIVC found",
             loc="left", fontsize="small")
ax.legend(fontsize="small", loc="upper left")
fig.tight_layout()

The front tells us *what* to trade; the next figure tells us *how* — the **anatomy** of each non-dominated plan. For every plan on the archive we draw the per-screen extraction rate as a stacked bar, ordered along the front. Reading left to right shows exactly how the optimizer reshapes the well's vertical draw as the priorities shift between more water and cleaner delivered blend.

In [ ]:
# per-screen extraction (m3/d) for each archive plan = base rate x its multiplier
arc = archive.copy()
for dv in RATE_DVS:
    arc[dv + "_q"] = arc[dv] * SCREEN_BASE[dv.split("-")[-1]]
arc = arc.sort_values("p95_blend")           # order along the risk axis

fig, ax = plt.subplots(figsize=(8.2, 5))
x = np.arange(len(arc))
bottom = np.zeros(len(arc))
scr_colors = {"dv-rate-ly1": "C3", "dv-rate-ly3": "C0", "dv-rate-ly5": "C2"}
for dv in RATE_DVS:
    q = arc[dv + "_q"].values
    ax.bar(x, q, bottom=bottom, width=0.82, color=scr_colors[dv],
           label=f"{dv.split('-')[-1]} (base {SCREEN_BASE[dv.split('-')[-1]]:.0f})")
    bottom += q
# canonical total draw, for reference
ax.axhline(BASE_RATE_TOTAL, color="0.3", ls=":", lw=1.2,
           label=f"canonical total ({BASE_RATE_TOTAL:.0f} m$^3$/d)")
ax.set_xticks(x)
ax.set_xticklabels([f"{p:.1f}" for p in arc.p95_blend], rotation=0, fontsize="small")
ax.set_xlabel("archive plan, ordered by P95(delivered blend SO$_4$) (mg/L, left = cleanest)")
ax.set_ylabel("per-screen extraction rate (m$^3$/d)")
ax.set_title("Anatomy of the front: how the optimizer redistributes pumping vertically",
             loc="left", fontsize="small")
ax.legend(fontsize="small", ncol=2, loc="upper left")
fig.tight_layout()

## The result, read honestly

The front is the lesson, and it is not the textbook diagonal — it is a **single point**. DSIVC drove the whole population to one corner: **every screen at its maximum rate, switching on at the earliest day** (the archive table and the anatomy figure both show it). That one plan supplies the most water *and* holds the lowest P95(delivered blend) at the same time; the conditioned blend there (≈73–74 mg/L) is even a touch *below* the canonical operation's, while supplying roughly twice the volume. There is no trade-off to arbitrate.

Three things worth stating plainly, because the shape is itself the finding:

- **The objectives are aligned, not conflicting — and we tested that honestly.** We deliberately gave the optimizer the *operationally correct* forecast (the delivered blend, which a vertical redistribution can actually move) rather than the worst-screen maximum (which it cannot). Even so, the front collapsed: in this ASR system, pumping harder captures more of the injected freshwater and *dilutes* the redox-front sulfate, so more water and cleaner water point the same way. The vertical split *does* change the delivered blend at a fixed volume — the per-screen probe shows a real, if small, response — but since you are never forced to hold volume fixed, that allocation effect never becomes a Pareto trade-off. DSIVC reports the alignment because it is real.
- **The residual risk is aquifer uncertainty, not the schedule.** At any fixed plan the conditioned spread in delivered blend SO₄ (the parameter-field uncertainty, ~10 mg/L wide here) dwarfs what the decision moves it (~1 mg/L across the whole rate range in the probe). An operator reading this learns that the sulfate risk is set by what we still don't know about the aquifer, not by how the well is run. The lever that would move it is *more data* (the dataworth notebook) or a *genuinely wider decision space* (e.g. extraction pushed past the injected volume, into the native water — a different, more expensive sweep), not a cleverer pump split within these bounds.
- **The 90 mg/L lens reads cleanly.** The recommended plan sits well below the line, so it is *reliable* — a 95% posterior chance of holding the delivered blend under the contractual trigger — and it does so at roughly double the canonical volume. One red line across the result we already have; no re-run.

A note on method, since it is the point of this notebook. We reached this the honest way, twice over: we gave the emulator decision axes it could legitimately reason about (the training sweep spanning the *vertical* pumping distribution), and we pointed it at the forecast the decision can actually influence (the delivered blend, not the worst screen). A tempting shortcut — reading percentiles off the raw sweep with a local smoother, or optimizing a worst-screen forecast a redistribution can never lower — would have either manufactured a busy "front" out of estimator noise or hidden the real allocation response entirely. DSIVC's answer is the one the physics and the data actually support: **for this system and these bounds, the decision is easy and the uncertainty is not.** That is a decision-support result, not a disappointment — knowing an expensive optimization is unnecessary is worth as much as a knife-edge trade-off.

## Validation: does the full model agree at the optimum?

The plan DSIVC handed us is the *emulator's* answer, and the discipline of this whole series is *never trust an emulator you have not tested*. So we test it, once, the expensive way. We take the `base` posterior parameter field and run the genuine reactive-transport model **at the DSIVC-recommended plan** — the vertical rate split and switch-on the optimizer selected — and ask whether the emulator's conditioned forecast there brackets the truth.

This is a real held-out test: in the training sweep, the `base` field was run at the *canonical* operation (every screen at 1.0, switch 308), never at the recommended split. So the full model has never seen this (field, decision) pair. That single ~6-minute run is prebaked by `etc/run_dsivc_mou_validation.py`; we load its result and compare.

In [ ]:
val_json_path = sweep_dir / "mou_validation.json"
val_obs_path = sweep_dir / "mou_validation_obs.csv"
if not (val_json_path.exists() and val_obs_path.exists()):
    raise Exception(
        f"full-model validation artifact not found in {sweep_dir}\n"
        "bake it with:  python etc/run_dsivc_mou_validation.py")

with open(val_json_path) as f:
    val = json.load(f)
val_obs = pd.read_csv(val_obs_path, index_col=0).iloc[:, 0]
val_obs.index = val_obs.index.astype(str)
opt_plan = {d: float(val["decvars"][d]) for d in DECVARS}   # the validated plan

# the full model's DELIVERED blend peak at the recommended plan (same blend
# function, applied to the genuine per-screen breakthrough)
fom_blend = float(blend_peak_mgl(val_obs.to_frame().T, opt_plan).iloc[0])
fom_worst = float(val["peak_so4_mgL"])                       # worst-screen monitor
_split = "  ".join(f"{d.split('-')[-1]}={opt_plan[d]:.2f}" for d in RATE_DVS)
print(f"full-model run: base field @ (screens {_split}, "
      f"switch {opt_plan['dv-switch-day']:.0f})")
print(" -- a decision the field was NOT run at in the sweep (its sweep run was 1/1/1, 308)")
print(f"full-model DELIVERED blend peak: {fom_blend:.1f} mg/L "
      f"(worst-screen monitor: {fom_worst:.1f} mg/L)")

Now condition the emulator on the *same* decision and read back the full posterior forecast — the whole conditioned stack — then apply the identical blend function to its per-screen outputs, so we compare like-for-like: the emulator's conditioned *delivered blend* against the full model's delivered blend.

In [ ]:
# condition the emulator at the optimum; the inner IES leaves its posterior stack
# on disk as dsi.<iter>.obs.[csv|jcb] -- take the last iteration it produced
evaluate_plan(opt_plan)
iters = [(int(f.split(".")[1]), f) for f in os.listdir(mou_td)
         if f.startswith("dsi.") and f.split(".")[1].isdigit()
         and (f.endswith(".obs.csv") or f.endswith(".obs.jcb"))]
_it, cond_file = max(iters)
if cond_file.endswith(".jcb"):
    cond = pyemu.ObservationEnsemble.from_binary(pst=dsi_pst, filename=str(mou_td / cond_file))._df
else:
    cond = pyemu.ObservationEnsemble.from_csv(pst=dsi_pst, filename=str(mou_td / cond_file))._df

# emulator's conditioned delivered-blend distribution, same blend function as the FOM
cond_blend = blend_peak_mgl(cond, opt_plan)
lo, med, hi = cond_blend.quantile([0.05, 0.5, 0.95])
inside = lo <= fom_blend <= hi
print(f"DSIVC-conditioned delivered blend at the optimum (mg/L): "
      f"P5={lo:.1f}  median={med:.1f}  P95={hi:.1f}")
print(f"full-model delivered blend:                             {fom_blend:.1f} mg/L")
print(f"full-model run falls {'INSIDE' if inside else 'OUTSIDE'} "
      f"the emulator's 5-95 conditioned band")

In [ ]:
# delivered-blend time series (mg/L) over the on-times, for an ensemble at a plan
def blend_series_mgl(ensemble, plan):
    q = plan_rates(plan); tot = sum(q.values())
    on = FORE_TIME >= plan["dv-switch-day"]
    blend = sum(q[s] * ensemble.loc[:, SCR_COLS[s]].values for s in SCREENS) / tot
    return FORE_TIME[on], blend[:, on] * 1000.0 * SO4_MW

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.6))

# --- left: the conditioned delivered-blend distribution, full-model run overlaid
axL.hist(cond_blend, bins=22, color="0.7", edgecolor="0.4",
         label="DSIVC-conditioned blend (stack)")
for q, c in [(lo, "0.3"), (med, "k"), (hi, "0.3")]:
    axL.axvline(q, color=c, ls=":", lw=1.2)
axL.axvline(fom_blend, color="crimson", lw=2.5, label=f"full model: {fom_blend:.1f} mg/L")
axL.axvline(CONTRACT_TRIGGER_MGL, color="orange", ls="--", lw=1.3,
            label=f"{CONTRACT_TRIGGER_MGL:.0f} mg/L trigger")
axL.set_xlabel("delivered blend peak SO$_4$ (mg/L)"); axL.set_ylabel("stack realisations")
axL.set_title("delivered-blend forecast at the optimum: emulator vs full model",
              loc="left", fontsize="small")
axL.legend(fontsize="x-small", loc="upper right")

# --- right: delivered-blend breakthrough, emulated envelope vs full-model line
tt, env = blend_series_mgl(cond, opt_plan)
_, fom_series = blend_series_mgl(val_obs.to_frame().T, opt_plan)
axR.fill_between(tt, np.quantile(env, 0.05, axis=0), np.quantile(env, 0.95, axis=0),
                 color="0.75", alpha=0.7, label="DSIVC conditioned P5-P95")
axR.plot(tt, np.quantile(env, 0.5, axis=0), color="0.35", lw=1.2, label="conditioned median")
axR.plot(tt, fom_series[0], color="crimson", lw=2, marker="o", ms=3,
         label="full model (base field)")
axR.set_xlabel("time (d)"); axR.set_ylabel("delivered blend SO$_4$ (mg/L)")
axR.set_title("delivered-blend breakthrough at the DSIVC-recommended plan",
              loc="left", fontsize="small")
axR.legend(fontsize="x-small", loc="upper left")
fig.tight_layout()

Read the two panels together. The genuine full-model **delivered blend** (the red line, left) sits **inside** the emulator's conditioned 5–95 band, and the full-model blend breakthrough (red, right) tracks **within** the emulator's conditioned envelope across the supply window. The emulator was not merely self-consistent — its forecast of the quantity that actually matters operationally, *at a decision it reached by optimization for a field-and-decision pair the full model had never run*, is borne out when we finally spend the six minutes to check.

That is the validation the optimization answer needed. DSIVC searched a vertical-distribution decision the base emulator never had, using a sweep built for exactly that purpose; the fidelity check confirms the axis was honestly learned. The caveat is the same one that governs the whole series: this holds across the decision bounds the sweep covered, and a single full-model run tests one point on the front — a decisive point, the recommended plan, but one point. Widen the decision space and the emulator must be re-swept and re-tested before its answer is believed.

## Wrap-up: what the emulation-first chain delivered

This is the last notebook in the sequence, so the wrap-up is the curriculum's wrap-up.

What *this* notebook bought us:

- It turned the forecast from a *verdict* on one fixed plan into a *search* over a four-dimensional decision space — three per-screen pumping rates and the switch-on day — each candidate conditioned on the emulator with its uncertainty honestly carried through, and handed back the non-dominated set plus the 90 mg/L lens from one figure.
- It got there only by asking the *right* question of the decision: what the well **delivers** is a flow-weighted blend, not the worst single screen, and it is the blend that the *vertical distribution* of pumping can actually move. Against a maximum-over-screens forecast the vertical lever sits inert (you cannot lower a maximum by reallocating); against the delivered blend it becomes a genuine handle, and the optimizer used it to trade a little volume for cleaner water where the physics allowed — while the residual risk stayed governed by aquifer uncertainty, not the schedule.
- It did so with **no full-model runs in the loop**. The only full-model cost was the one-off training sweep; the emulator carried the entire optimization.

And the compute receipts for the whole chain, which are the thesis of the series stated in numbers:

In [ ]:
RUN_MIN = 6   # minutes per full-model DIZON run

prior_mc_runs = 201          # part1_04 prior Monte Carlo
full_ies_runs = 201 * 4      # part1_06 full-model IES (~4 iterations)
sweep_runs = len(run_ids)    # this notebook's training sweep

def days(n): return n * RUN_MIN / 60 / 24
print("FULL-MODEL COMPUTE PAID (once, by the maintainer):")
print(f"  prior Monte Carlo:        ~{prior_mc_runs} runs  (~{days(prior_mc_runs):.1f} d serial)")
print(f"  full-model history match: ~{full_ies_runs} runs  (~{days(full_ies_runs):.1f} d serial)")
print(f"  decision-variable sweep:  ~{sweep_runs} runs  (~{days(sweep_runs):.1f} d serial)")
print(f"  -- and that is ALL the full model ever ran.\n")
print("EMULATOR-SIDE, FOR ALMOST NOTHING MORE:")
print("  DSI conditioning (part1_05):      seconds-minutes")
print("  posterior calibration audit:      minutes")
print("  dataworth, all arms (part1_07):   minutes")
print("  DSIVC optimization (part1_08):    this whole search ran on the emulator")

The asymmetry is the point. A reactive-transport model costs minutes per run; a history match, a dataworth study, and an optimization each want hundreds to thousands of runs *inside* an iterative loop. Run the full model inside those loops and the curriculum is months of compute. Run it once to build the ensembles the emulator learns from — prior MC, history match, decision sweep — and every downstream question (condition, validate, value data, optimize) is answered on the emulator for seconds to minutes more.

The caveat that earns its keep, restated one last time: **the emulator can only reason about what it was trained on.** The DSI posterior was trustworthy because we validated it against held-out realisations *before* conditioning. This optimization is trustworthy across the decision-variable bounds the sweep covered, and no further — widening the bounds means a new sweep, and another fidelity check before the answer is believed. Same principle, all the way down: never trust an emulator you have not tested, on a question it has not seen.